# 📄 Notebook 2 — PDF Parser
## What this notebook does
Reads your contract PDF files and extracts all the text from them.

**Simple explanation:**  
Like copy-pasting a PDF into Notepad — but automated for every page.

**Technical explanation:**  
Uses PyMuPDF (fitz) to iterate over PDF pages and extract text blocks.  
Uses pdfplumber as a fallback for PDFs with complex table layouts.  
Splits extracted text into overlapping chunks using LangChain's  
RecursiveCharacterTextSplitter (chunk_size=500, overlap=50 tokens).


## Step 1 — Import Libraries

In [ ]:
import fitz          # PyMuPDF — main PDF reader
import pdfplumber    # fallback for table-heavy PDFs
import os
import json
from pathlib import Path
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

# Connect to database
engine  = create_engine("sqlite:///database/contractiq.db", echo=False)
Session = sessionmaker(bind=engine)
session = Session()

print("✅ Libraries loaded")
print(f"✅ PyMuPDF version: {fitz.__version__}")

## Step 2 — The PDF Text Extractor Function

**What it does:** Opens a PDF, reads every page, returns clean text.

**Why two libraries?** PyMuPDF is fast. pdfplumber is better at tables. We use both.

In [ ]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts all text from a PDF file.
    
    Simple:    Opens every page, reads all words, joins them together.
    Technical: Uses PyMuPDF's page.get_text() with 'blocks' mode for
               structured extraction. Falls back to pdfplumber for
               pages where PyMuPDF returns less than 50 characters.
    
    Returns: dict with filename, pages list, full_text, page_count
    """
    path     = Path(pdf_path)
    result   = {
        "filename"  : path.name,
        "filepath"  : str(path),
        "pages"     : [],
        "full_text" : "",
        "page_count": 0
    }
    
    # ── Method 1: PyMuPDF (fast, handles most PDFs) ──────────
    try:
        doc = fitz.open(pdf_path)
        result["page_count"] = len(doc)
        
        for page_num in range(len(doc)):
            page      = doc[page_num]
            page_text = page.get_text("text")  # extract as plain text
            
            # ── Fallback: pdfplumber for complex pages ────────
            if len(page_text.strip()) < 50:
                with pdfplumber.open(pdf_path) as plumber_doc:
                    if page_num < len(plumber_doc.pages):
                        plumber_page = plumber_doc.pages[page_num]
                        page_text    = plumber_page.extract_text() or ""
                        
                        # also extract any tables on this page
                        tables = plumber_page.extract_tables()
                        for table in tables:
                            for row in table:
                                if row:
                                    page_text += " | ".join(
                                        [str(cell) for cell in row if cell]
                                    ) + "\n"
            
            result["pages"].append({
                "page_num": page_num + 1,
                "text"    : page_text.strip()
            })
        
        doc.close()
        result["full_text"] = "\n\n".join(
            [p["text"] for p in result["pages"] if p["text"]]
        )
        
        print(f"✅ Extracted: {path.name}")
        print(f"   Pages: {result['page_count']}")
        print(f"   Characters: {len(result['full_text']):,}")
        
    except Exception as e:
        print(f"❌ Error reading {path.name}: {e}")
    
    return result

print("✅ extract_text_from_pdf() function defined")

## Step 3 — The Text Chunker Function

**Simple:** Cuts the big contract text into small pieces the AI can read.

**Technical:** Uses LangChain's `RecursiveCharacterTextSplitter`. Chunk size = 1000 chars, overlap = 100 chars. Overlap ensures obligations at chunk boundaries are not lost.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def chunk_text(text, doc_name, chunk_size=1000, overlap=100):
    """
    Splits large text into smaller overlapping chunks.
    
    Simple:    Imagine cutting a long newspaper into small cards,
               with each card sharing a few lines with the next one.
               That way nothing important falls between two cards.
    
    Technical: RecursiveCharacterTextSplitter tries to split on
               paragraph breaks first, then sentences, then words.
               This preserves semantic units better than hard splits.
    
    Returns: list of dicts with chunk_text, chunk_index, doc_name
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size        = chunk_size,
        chunk_overlap     = overlap,
        separators        = ["\n\n", "\n", ". ", " ", ""],
        length_function   = len,
    )
    
    raw_chunks = splitter.split_text(text)
    
    chunks = []
    for i, chunk in enumerate(raw_chunks):
        if len(chunk.strip()) > 50:   # skip very small chunks
            chunks.append({
                "chunk_text"  : chunk.strip(),
                "chunk_index" : i,
                "doc_name"    : doc_name,
            })
    
    print(f"✅ {doc_name}: {len(chunks)} chunks created")
    print(f"   Avg chunk size: {sum(len(c['chunk_text']) for c in chunks) // len(chunks) if chunks else 0} chars")
    return chunks

print("✅ chunk_text() function defined")

## Step 4 — Process All Contracts

Put your contract PDFs in the `test_contracts/` folder.
This cell reads all of them and saves to the database.

In [ ]:
from sqlalchemy import text as sql_text
import sys
sys.path.insert(0, '.')

# ── Recreate DB models inline ─────────────────────────────────
from sqlalchemy import Column, Integer, String, Text, DateTime, Float
from sqlalchemy.orm import declarative_base
from datetime import datetime

Base2   = declarative_base()

class Document(Base2):
    __tablename__ = "documents"
    __table_args__ = {'extend_existing': True}
    id           = Column(Integer, primary_key=True)
    filename     = Column(String(255))
    filepath     = Column(String(500))
    upload_date  = Column(DateTime, default=datetime.now)
    total_pages  = Column(Integer, default=0)
    total_chunks = Column(Integer, default=0)
    status       = Column(String(50), default="uploaded")

class Chunk(Base2):
    __tablename__ = "chunks"
    __table_args__ = {'extend_existing': True}
    id           = Column(Integer, primary_key=True)
    doc_id       = Column(Integer)
    doc_name     = Column(String(255))
    chunk_text   = Column(Text)
    page_number  = Column(Integer, default=0)
    chunk_index  = Column(Integer, default=0)

Base2.metadata.create_all(engine)

# ── Find all PDFs ─────────────────────────────────────────────
pdf_folder    = "test_contracts"
all_pdfs      = list(Path(pdf_folder).glob("*.pdf"))

if not all_pdfs:
    print(f"⚠️  No PDFs found in '{pdf_folder}/' folder!")
    print("   Copy your generated contracts there first:")
    print("   e.g. copy contracts/run_TIMESTAMP/*.pdf test_contracts/")
else:
    print(f"📂 Found {len(all_pdfs)} PDF files:\n")
    
    all_chunks = []   # collect all chunks for later use
    
    for pdf_path in all_pdfs:
        # 1. Extract text
        result = extract_text_from_pdf(str(pdf_path))
        
        if not result["full_text"]:
            continue
        
        # 2. Save document to DB
        existing = session.query(Document).filter_by(
            filename=result["filename"]
        ).first()
        
        if existing:
            doc_record = existing
            print(f"   (already in DB — updating)")
        else:
            doc_record = Document(
                filename   = result["filename"],
                filepath   = result["filepath"],
                total_pages= result["page_count"],
                status     = "processing"
            )
            session.add(doc_record)
            session.flush()   # get the ID
        
        # 3. Chunk the text
        chunks = chunk_text(result["full_text"], result["filename"])
        
        # 4. Save chunks to DB
        for chunk in chunks:
            existing_chunk = session.query(Chunk).filter_by(
                doc_id=doc_record.id,
                chunk_index=chunk["chunk_index"]
            ).first()
            if not existing_chunk:
                session.add(Chunk(
                    doc_id     = doc_record.id,
                    doc_name   = chunk["doc_name"],
                    chunk_text = chunk["chunk_text"],
                    chunk_index= chunk["chunk_index"],
                ))
        
        # 5. Update doc status
        doc_record.total_chunks = len(chunks)
        doc_record.status       = "parsed"
        
        all_chunks.extend(chunks)
        print()
    
    session.commit()
    
    # Summary
    total_docs   = session.query(Document).count()
    total_chunks = session.query(Chunk).count()
    print(f"\n{'='*50}")
    print(f"✅ PARSING COMPLETE")
    print(f"   Documents in database : {total_docs}")
    print(f"   Total chunks stored   : {total_chunks}")
    print(f"{'='*50}")
    print("\n▶ Run Notebook 3 next: Obligation Extractor")

## Step 5 — Verify What Was Stored

In [ ]:
import pandas as pd

# Show documents table
docs = session.execute(sql_text(
    "SELECT id, filename, total_pages, total_chunks, status FROM documents"
)).fetchall()

df_docs = pd.DataFrame(docs, columns=["ID","Filename","Pages","Chunks","Status"])
print("📋 DOCUMENTS TABLE:")
print(df_docs.to_string(index=False))

print()

# Show sample chunk
sample = session.execute(sql_text(
    "SELECT doc_name, chunk_index, substr(chunk_text,1,200) FROM chunks LIMIT 3"
)).fetchall()

print("\n📋 SAMPLE CHUNKS (first 200 chars):")
for row in sample:
    print(f"\n  Doc: {row[0]} | Chunk #{row[1]}")
    print(f"  Text: {row[2]}...")